# Synctra — All-in-One Colab

Runs the full Synctra stack inside a single Colab GPU runtime:

| Component | Port | Source |
|---|---|---|
| Trained NLP tool router | 8001 | `tool/colab_course_import_agent_server.py` + `tool/nlp_tool_calling_agent.py` |
| Qwen course-import LLM | 8001 (same) | same file |
| FastAPI backend (uvicorn) | 8000 | `backend/` |
| Cloudflared tunnel | — | exposes port 8000 to your Flutter app |

### Before running
- **Runtime → Change runtime type → T4 GPU** (or better).
- Have your GitHub repo URL ready, or be ready to upload files manually.

### Run cells top-to-bottom. The final cell blocks; keep the runtime open while your app is in use.

## 1. Install dependencies

In [ ]:
%pip -q install "torch>=2.2,<3" "transformers>=4.44,<5" "accelerate>=0.33,<2" "datasets>=2.19,<4" "sentencepiece>=0.2,<1" "bitsandbytes>=0.43" "numpy>=1.26,<3" "pandas>=2.2,<3" "pyarrow>=15,<24" "scikit-learn>=1.4,<2"
!pip -q install fastapi uvicorn pydantic pydantic-settings httpx python-dotenv
!pip -q install beautifulsoup4 supabase
print('Dependencies installed.')

## 2. Get the repo into Colab

**Option A** — clone from GitHub (recommended). Replace the URL with your fork if needed.

In [ ]:
REPO_URL = 'https://github.com/rpham03/synctraCapstone.git'
REPO_DIR = '/content/syntra'

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
print('Repo ready at', REPO_DIR)
!ls $REPO_DIR/tool/ | head

**Option B** — if your repo is private, run this cell instead and upload `synctra_capstone.zip` from the Files panel.

In [ ]:
# Uncomment to use upload instead of git clone:
#
# from google.colab import files
# uploaded = files.upload()  # pick synctra_capstone.zip
# import zipfile, shutil, os
# shutil.rmtree('/content/syntra', ignore_errors=True)
# with zipfile.ZipFile(list(uploaded.keys())[0]) as zf:
#     zf.extractall('/content/syntra')
# print(os.listdir('/content/syntra'))

## 3. Train the NLP tool router  *(skip if you already have a model)*

Fine-tunes both models on the balanced 5,000-row / 17-tool dataset using a deterministic 70% training / 30% held-out testing split. It records real transformer metrics from the held-out split, not the Naive Bayes proxy.

Outputs `/content/syntra_tool_router/`. On a T4 this can take several minutes. Skip this cell only if you previously saved this updated model and its metric JSON files to Drive.

In [ ]:
import subprocess
from pathlib import Path

TRAINER = Path('/content/syntra/tool/one_click_train_nlp_router_colab.py')
METRICS_SUMMARY = Path('/content/syntra_tool_router/model_metrics_summary.json')
trainer_source = TRAINER.read_text()
if 'model_metrics_summary.json' not in trainer_source:
    raise RuntimeError(
        'Colab cloned an older trainer that cannot create the metrics report. '
        'Push/pull the metrics changes, rerun cell 2, then rerun this cell.'
    )

TRAINING_LOG = Path('/content/syntra_nlp_training.log')
METRICS_SUMMARY.unlink(missing_ok=True)
command = [
    'python', '-u', str(TRAINER),
    '--dataset-size', '5000',
    '--test-ratio', '0.30',
    '--no-install-deps',
    '--skip-manual-eval-files',
]
print('Running:', ' '.join(command))
print('Full training log:', TRAINING_LOG)
with TRAINING_LOG.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code:
    tail = '\n'.join(TRAINING_LOG.read_text(errors='replace').splitlines()[-120:])
    if METRICS_SUMMARY.exists():
        print(
            f'Warning: a post-metrics optional step failed with exit code {return_code}, '
            f'but the current training run created {METRICS_SUMMARY}.\n\n{tail}'
        )
    else:
        raise RuntimeError(
            f'NLP training failed with exit code {return_code}. '
            f'Last log lines from {TRAINING_LOG}:\n\n{tail}'
        )

if not METRICS_SUMMARY.exists():
    raise RuntimeError(
        f'Training exited without creating {METRICS_SUMMARY}. '
        'Read the training output above for the first error.'
    )
print('Training and metrics completed:', METRICS_SUMMARY)

### 3a. Real trained-model metrics and downloadable report

Run this after training. It displays the exact held-out metrics produced during the training run:

- intent model: overall accuracy, macro/weighted F1, all 17 per-tool metrics, confidence calibration, confusion matrix
- slot model: entity precision/recall/F1, per-slot metrics, exact-sequence accuracy, token accuracy
- both models: parameter count, artifact size, dtype/device, GPU/runtime versions, training runtime, and single-prompt latency p50/p95/p99

The cell also creates `/content/syntra_nlp_metrics.zip` for your report or reviewer evidence.

In [ ]:
import json, shutil
from pathlib import Path
import pandas as pd
from IPython.display import display

MODEL_DIR = Path('/content/syntra_tool_router')
SUMMARY_PATH = MODEL_DIR / 'model_metrics_summary.json'
if not SUMMARY_PATH.exists():
    existing = sorted(str(path.relative_to(MODEL_DIR)) for path in MODEL_DIR.rglob('*') if path.is_file()) if MODEL_DIR.exists() else []
    raise FileNotFoundError(
        f'Missing {SUMMARY_PATH}. Cell 3 did not finish with the updated trainer. '
        f'Files currently under {MODEL_DIR}: {existing or "none"}. '
        'Rerun the repo pull cell, then rerun cell 3 and fix the first error shown there.'
    )
summary = json.loads(SUMMARY_PATH.read_text())
intent = summary['intent_model']
slot = summary.get('slot_model')
intent_test_hash = intent['split'].get('test_text_sha256')
slot_test_hash = slot['split'].get('test_text_sha256') if slot else None

print('=== Reviewer-ready model summary ===')
print(json.dumps({
    'approach': 'hybrid: trained 17-label classifier + trained slot extractor + deterministic safety rules',
    'dataset_rows': intent['split']['total_examples'],
    'train_rows': intent['split']['train_examples'],
    'held_out_test_rows': intent['split']['test_examples'],
    'split_seed': intent['split']['seed'],
    'shared_intent_slot_test_split_verified': bool(intent_test_hash and slot_test_hash and intent_test_hash == slot_test_hash),
    'intent_labels': len(intent['per_tool']),
    'intent_accuracy': intent['accuracy'],
    'intent_macro_f1': intent['macro_average']['f1'],
    'intent_weighted_f1': intent['weighted_average']['f1'],
    'intent_parameters': intent['model']['total_parameters'],
    'intent_latency_p50_ms': intent['latency']['p50_ms'],
    'slot_entity_micro_f1': slot['entity_micro']['f1'] if slot else None,
    'slot_exact_sequence_accuracy': slot['exact_sequence_accuracy'] if slot else None,
    'slot_parameters': slot['model']['total_parameters'] if slot else None,
    'slot_latency_p50_ms': slot['latency']['p50_ms'] if slot else None,
    'combined_parameters': intent['model']['total_parameters'] + (slot['model']['total_parameters'] if slot else 0),
}, indent=2))

print('\n=== Intent model information ===')
display(pd.DataFrame([intent['model']]).T.rename(columns={0: 'value'}))
print('\n=== Intent training and latency ===')
display(pd.DataFrame([intent['training'], intent['latency']], index=['training', 'latency']).T)
intent_history = pd.DataFrame(intent.get('training_history', []))
if not intent_history.empty:
    history_columns = [name for name in ['epoch', 'step', 'loss', 'eval_loss', 'eval_accuracy', 'grad_norm', 'learning_rate'] if name in intent_history.columns]
    print('\n=== Intent training history ===')
    display(intent_history[history_columns])

intent_rows = [dict(tool=tool, **values) for tool, values in intent['per_tool'].items()]
print('\n=== Real held-out per-tool metrics (worst F1 first) ===')
display(pd.DataFrame(intent_rows).sort_values(['f1', 'accuracy', 'tool']).reset_index(drop=True))
print('\nPer-tool accuracy definition:', intent['metric_definitions']['per_tool_accuracy'])

labels = intent['confusion_matrix']['labels']
matrix = intent['confusion_matrix']['rows_expected_columns_predicted']
print('\n=== Confusion matrix: rows=expected, columns=predicted ===')
display(pd.DataFrame(matrix, index=labels, columns=labels))

print('\n=== Confidence and calibration ===')
display(pd.DataFrame([{k: v for k, v in intent['confidence'].items() if k != 'bins'}]))

if slot:
    print('\n=== Slot model information ===')
    display(pd.DataFrame([slot['model']]).T.rename(columns={0: 'value'}))
    print('\n=== Slot model held-out summary ===')
    display(pd.DataFrame([{
        'entity_micro_precision': slot['entity_micro']['precision'],
        'entity_micro_recall': slot['entity_micro']['recall'],
        'entity_micro_f1': slot['entity_micro']['f1'],
        'entity_macro_f1': slot['entity_macro_f1'],
        'exact_sequence_accuracy': slot['exact_sequence_accuracy'],
        'token_accuracy_including_o': slot['token_accuracy_including_o'],
        'non_o_token_accuracy': slot['non_o_token_accuracy'],
        **{f'latency_{key}': value for key, value in slot['latency'].items()},
    }]))
    slot_history = pd.DataFrame(slot.get('training_history', []))
    if not slot_history.empty:
        history_columns = [name for name in ['epoch', 'step', 'loss', 'eval_loss', 'eval_token_accuracy', 'grad_norm', 'learning_rate'] if name in slot_history.columns]
        print('\n=== Slot training history ===')
        display(slot_history[history_columns])
    slot_rows = [dict(slot=name, **values) for name, values in slot['per_slot'].items()]
    print('\n=== Real held-out per-slot entity metrics ===')
    display(pd.DataFrame(slot_rows).sort_values(['f1', 'slot']).reset_index(drop=True))

metrics_dir = Path('/content/syntra_nlp_metrics')
shutil.rmtree(metrics_dir, ignore_errors=True)
metrics_dir.mkdir()
for path in [
    MODEL_DIR / 'model_metrics_summary.json',
    MODEL_DIR / 'intent_metrics.json',
    MODEL_DIR / 'intent_confusion_matrix.csv',
    MODEL_DIR / 'intent_test_predictions.jsonl',
    MODEL_DIR / 'training_meta.json',
    MODEL_DIR / 'slot_model' / 'slot_metrics.json',
    MODEL_DIR / 'slot_model' / 'training_meta.json',
]:
    if path.exists():
        shutil.copy2(path, metrics_dir / path.name.replace('training_meta', f'{path.parent.name}_training_meta'))
archive = shutil.make_archive('/content/syntra_nlp_metrics', 'zip', metrics_dir)
print('\nMetrics archive ready:', archive)
print('Artifact files:', sorted(path.name for path in metrics_dir.iterdir()))

### 3b. (Optional) Persist the trained model to Google Drive

Colab wipes `/content/` between sessions. Save the model to Drive once, then restore it later instead of retraining.

In [ ]:
# Mount Drive and copy the trained model.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/syntra_tool_router'
LOCAL_MODEL_DIR = '/content/syntra_tool_router'

if os.path.isdir(LOCAL_MODEL_DIR) and not os.path.isdir(DRIVE_MODEL_DIR):
    shutil.copytree(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR)
    print('Saved model to Drive at', DRIVE_MODEL_DIR)
elif os.path.isdir(DRIVE_MODEL_DIR) and not os.path.isdir(LOCAL_MODEL_DIR):
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    print('Restored model from Drive')
else:
    print('Local model present:', os.path.isdir(LOCAL_MODEL_DIR),
          '| Drive model present:', os.path.isdir(DRIVE_MODEL_DIR))

## 4. Start the model server (port 8001) in the background

Loads Qwen (course import + ai_agent fallback) and the trained classifier (chat router). Wait ~60 seconds for the model to download/load before continuing.

In [ ]:
import subprocess, time, os, sys, urllib.request, json

TOOL_DIR = '/content/syntra/tool'
MODEL_DIR = '/content/syntra_tool_router'

assert os.path.isdir(MODEL_DIR), f'No trained model at {MODEL_DIR}. Train it first or restore from Drive.'

model_server_log = open('/content/model_server.log', 'wb')
model_server = subprocess.Popen(
    [
        sys.executable, f'{TOOL_DIR}/colab_course_import_agent_server.py',
        '--host', '127.0.0.1',
        '--port', '8001',
        '--tunnel', 'none',
        '--nlp-router-model-dir', MODEL_DIR,
        '--nlp-agent-path', TOOL_DIR,
    ],
    stdout=model_server_log,
    stderr=subprocess.STDOUT,
)
print('Model server PID:', model_server.pid)

for attempt in range(120):
    time.sleep(2)
    try:
        with urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=2) as r:
            data = json.loads(r.read())
        print('Model server up:', json.dumps(data, indent=2))
        break
    except Exception:
        if attempt % 10 == 0:
            print(f'... waiting for model server ({attempt*2}s)')
else:
    raise RuntimeError('Model server did not come up. Check /content/model_server.log')

In [ ]:
# Smoke-test that /api/generate (ai_agent) responds locally.
# Run this after cell 12; if it prints an answer, COLAB_AI_AGENT_HOST=http://127.0.0.1:8001 will work end-to-end.
import json, urllib.request

req = urllib.request.Request(
    'http://127.0.0.1:8001/api/generate',
    data=json.dumps({
        'prompt': 'hi',
        'stream': False,
        'options': {'syntra_mode': 'ai_agent', 'temperature': 0.2},
    }).encode(),
    headers={'Content-Type': 'application/json'},
)
with urllib.request.urlopen(req, timeout=120) as r:
    body = json.loads(r.read())
print('ai_agent response:', body.get('response', '')[:400])

## 4b. (Optional) Expose port 8001 via your reserved ngrok domain

Only needed if the FastAPI backend runs **outside this Colab** (e.g. on your laptop) and needs to reach the NLP router over the internet. Skip if you're running the backend in this same notebook (cells 17–18) — loopback `http://127.0.0.1:8001` is faster.

**Setup once:**
1. Rotate / generate an ngrok authtoken at https://dashboard.ngrok.com/.
2. In this Colab, click the 🔑 key icon in the left sidebar → **Add new secret**.
3. Name it `NGROK_AUTHTOKEN`, paste the token, toggle "Notebook access" on.

The token is read from the Colab secret store at runtime — never hard-code it in this notebook or commit it.

In [ ]:
!pip -q install pyngrok

from google.colab import userdata
from pyngrok import ngrok, conf

NGROK_DOMAIN = 'wieldable-prowess-recognize.ngrok-free.dev'

try:
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    token = None
if not token:
    raise RuntimeError(
        'Add a Colab secret named NGROK_AUTHTOKEN (key icon in the left sidebar), '
        'then re-run this cell.'
    )

conf.get_default().auth_token = token

# Make this cell idempotent — drop any tunnels left over from a previous run.
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public = ngrok.connect(8001, 'http', domain=NGROK_DOMAIN)
print('NLP router public URL:', public.public_url)
print('Point your local backend/.env at:')
print(f'  COLAB_NLP_ROUTER_HOST={public.public_url}')

## 5. Configure the Synctra backend

- Points chat at the trained Colab NLP router.
- Writes `CHAT_LLM_PROVIDER=nlp` and `COLAB_NLP_ROUTER_HOST=...` into `/content/syntra/backend/.env`.
- Run this cell before starting uvicorn.
- Keeps `OLLAMA_HOST` available only as a fallback for older course-import wiring.
- Registers the `chat_colab` route in `main.py` if not already wired up.

In [ ]:
import os, pathlib

BACKEND_DIR = pathlib.Path('/content/syntra/backend')
# The model server (cell 12) runs on 127.0.0.1:8001 inside this same Colab VM,
# so the backend talks to it over loopback — no public tunnel required.
# Both /plan (NLP router) and /api/generate (ai_agent) are served on that port.
# Only override these if you split the model server across a different runtime.
NLP_ROUTER_HOST = os.environ.get(
    'COLAB_NLP_ROUTER_HOST',
    'http://127.0.0.1:8001',
).strip()
AI_AGENT_HOST = (
    os.environ.get('COLAB_AI_AGENT_HOST')
    or os.environ.get('COLAB_COURSE_IMPORT_HOST')
    or 'http://127.0.0.1:8001'
).strip()
if not NLP_ROUTER_HOST:
    raise RuntimeError('Set COLAB_NLP_ROUTER_HOST to your NLP router URL first.')

env_updates = {
    'CHAT_LLM_PROVIDER': 'nlp',
    'COLAB_NLP_ROUTER_HOST': NLP_ROUTER_HOST,
    'COLAB_AI_AGENT_HOST': AI_AGENT_HOST,
    'COLAB_COURSE_IMPORT_HOST': AI_AGENT_HOST,
    'OLLAMA_HOST': 'http://127.0.0.1:8001',
}

for key, value in env_updates.items():
    os.environ[key] = value

# Write/update .env so settings.py also picks it up before uvicorn starts.
env_file = BACKEND_DIR / '.env'
existing_lines = env_file.read_text().splitlines() if env_file.exists() else []
kept_lines = [line for line in existing_lines if line.split('=', 1)[0] not in env_updates]
new_lines = kept_lines + [f'{key}={value}' for key, value in env_updates.items()]
env_file.write_text('\n'.join(new_lines).rstrip() + '\n')
print('Configured backend chat NLP router in', env_file)
for line in env_file.read_text().splitlines():
    if line.startswith(('CHAT_LLM_PROVIDER=', 'COLAB_NLP_ROUTER_HOST=', 'COLAB_AI_AGENT_HOST=', 'COLAB_COURSE_IMPORT_HOST=')):
        print(line)

# Repair/register backend routes in main.py.
main_py = BACKEND_DIR / 'app/main.py'
main_py.write_text('''# FastAPI application entry point — registers all API routers and CORS middleware.
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from app.api.v1.routes import (
    auth,
    canvas,
    chat,
    collab,
    chat_colab,
    course_import_colab,
    events,
    schedule,
    tasks,
)

app = FastAPI(title="Syntra API", version="0.1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(auth.router,     prefix="/api/v1/auth",     tags=["auth"])
app.include_router(events.router,   prefix="/api/v1/events",   tags=["events"])
app.include_router(tasks.router,    prefix="/api/v1/tasks",    tags=["tasks"])
app.include_router(schedule.router, prefix="/api/v1/schedule", tags=["schedule"])
app.include_router(chat.router,     prefix="/api/v1/chat",     tags=["chat"])
app.include_router(collab.router,   prefix="/api/v1/collab",   tags=["collab"])
app.include_router(chat_colab.router, prefix="/api/v1/chat-colab", tags=["chat-colab"])
app.include_router(canvas.router,   prefix="/api/v1/canvas",   tags=["canvas"])
app.include_router(
    course_import_colab.router,
    prefix="/api/v1/course-import",
    tags=["course-import"],
)
''')
print('Rewrote app/main.py with valid chat and chat_colab routes')

# Install the backend requirements.
!pip -q install -r {BACKEND_DIR}/requirements.txt

## 6. Open a cloudflared tunnel for the backend (port 8000)

The printed `https://*.trycloudflare.com` URL is what you point your Flutter app at.

This tunnel only forwards traffic. The backend is not usable until the uvicorn cell in Section 7 is also running.

In [ ]:
import sys
sys.path.insert(0, '/content/syntra/tool')
from colab_course_import_agent_server import start_cloudflared_tunnel
start_cloudflared_tunnel(8000, wait_for_url_s=60, backend_app=True)
print('Cloudflared tunnel cell finished. Use the public URL printed above.')

## 7. Launch the FastAPI backend (uvicorn on port 8000)

This cell **blocks** — it must stay running for your app to work. To stop, click the stop button on this cell.

If Flutter says it cannot reach the backend, this cell is stopped, the Cloudflare tunnel cell is stopped, or Flutter was launched with an old tunnel URL.

In [ ]:
%cd /content/syntra/backend
!uvicorn app.main:app --host 0.0.0.0 --port 8000

## 8. (Optional) Quick smoke test — run from a *separate* Colab cell or local terminal

While the uvicorn cell above keeps running, you can test from another browser tab:

```bash
curl https://YOUR_TUNNEL.trycloudflare.com/api/v1/chat-colab/health | jq

curl -X POST https://YOUR_TUNNEL.trycloudflare.com/api/v1/chat-colab/ \
  -H 'Content-Type: application/json' \
  -d '{"message":"what homework is due this week"}' | jq
```

Expected: the classifier routes to `get_tasks`, the backend executes it, and you get a JSON response.

If something fails, check `/content/model_server.log` for model server errors.